# Cardio Catch Disease — 04. Modelling & Business Results

> **Hands-On ML principle (Ch. 2, 3, 6):** Try multiple models, use cross-validation to compare them fairly, tune the best one with RandomizedSearchCV, then evaluate **once** on the test set. Translate ML metrics into business impact.

---

## 0. Setup

In [ ]:
from pathlib import Path
import sys, warnings, json

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore")

FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

INTERIM = PROJECT_ROOT / "data" / "interim"

## 1. Load Prepared Data from Notebook 03

If the interim files don't exist, re-run Notebook 03 first or use the automated pipeline.

In [ ]:
prepared_path = INTERIM / "prepared_data.joblib"

if prepared_path.exists():
    X_train, y_train, X_test, y_test = joblib.load(prepared_path)
    print(f"Loaded prepared data from {prepared_path}")
else:
    # Fallback: build inline (same logic as Notebook 03)
    print("Prepared data not found — building inline...")
    from cardio_catch_disease.data import load_training_frame
    from cardio_catch_disease.config import load_config
    from cardio_catch_disease.features import prepare_features, model_matrix
    from sklearn.model_selection import train_test_split

    config = load_config(PROJECT_ROOT / "configs" / "project.toml")
    df = load_training_frame(config)
    train_set, test_set = train_test_split(df, test_size=0.20, random_state=42, stratify=df["cardio"])

    X_train_raw, y_train, _ = model_matrix(prepare_features(train_set, config), config)
    X_test_raw,  y_test,  _ = model_matrix(prepare_features(test_set,  config), config)

    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler, OneHotEncoder

    cat_cols = X_train_raw.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X_train_raw.columns if c not in cat_cols]

    preprocessor = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc",  StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ])

    X_train = preprocessor.fit_transform(X_train_raw)
    X_test  = preprocessor.transform(X_test_raw)
    print("Done.")

print(f"\nX_train: {X_train.shape}  |  y_train positives: {y_train.mean():.1%}")
print(f"X_test:  {X_test.shape}   |  y_test positives:  {y_test.mean():.1%}")

## 2. Baseline Model — Logistic Regression

> **Book principle (Ch. 2):** Always start with a simple baseline. It tells you what a reasonable model is *without* any tuning and sets the floor for comparison.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

lr_baseline = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42, n_jobs=-1)
lr_scores = cross_val_score(lr_baseline, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)

print("Logistic Regression — 5-fold Cross-Validation (ROC-AUC):")
print(f"  Scores : {[f'{s:.4f}' for s in lr_scores]}")
print(f"  Mean   : {lr_scores.mean():.4f}")
print(f"  Std    : {lr_scores.std():.4f}")

## 3. Model Comparison — Cross-Validation

> **Book principle (Ch. 2):** Try multiple models from different families to find the best one before tuning. Cross-validation gives a more reliable estimate than a single train/validation split.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

candidates = {
    "Logistic Regression":     LogisticRegression(max_iter=1000, class_weight="balanced",
                                                   random_state=42, n_jobs=-1),
    "Random Forest":           RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                                      random_state=42, n_jobs=-1),
    "Gradient Boosting":       GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# Try to add XGBoost if installed
try:
    from xgboost import XGBClassifier
    candidates["XGBoost"] = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                                           use_label_encoder=False, eval_metric="logloss")
    print("XGBoost available — added to comparison.")
except ImportError:
    print("XGBoost not installed — skipping. (pip install xgboost to include)")

cv_results = {}
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
    cv_results[name] = {"mean": scores.mean(), "std": scores.std(), "scores": scores}
    print(f"  {name:<30} ROC-AUC = {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Visualise comparison
names = list(cv_results.keys())
means = [cv_results[n]["mean"] for n in names]
stds  = [cv_results[n]["std"]  for n in names]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#e74c3c" if m == max(means) else "#3498db" for m in means]
bars = ax.barh(names, means, xerr=stds, color=colors, edgecolor="white", capsize=4)
ax.set_xlabel("ROC-AUC (5-fold CV)", fontsize=12)
ax.set_title("Model Comparison — Cross-Validation ROC-AUC", fontsize=13, fontweight="bold")
ax.axvline(x=max(means), color="red", linestyle="--", alpha=0.4)
ax.set_xlim(0.6, 0.85)
for bar, mean in zip(bars, means):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{mean:.4f}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES / "04_model_comparison.png", bbox_inches="tight")
plt.show()

best_name = max(cv_results, key=lambda k: cv_results[k]["mean"])
print(f"\nBest model: {best_name} (ROC-AUC = {cv_results[best_name]['mean']:.4f})")

## 4. Hyperparameter Tuning — RandomizedSearchCV

> **Book principle (Ch. 2):** "Once you have a shortlist of promising models, tune their hyperparameters using cross-validation. Use RandomizedSearchCV rather than GridSearchCV when the hyperparameter space is large."

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators":    [100, 200, 300, 400],
    "max_depth":       [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":    ["sqrt", "log2", 0.5],
    "class_weight":    ["balanced", None],
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rnd_search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=30, cv=5, scoring="roc_auc",
    random_state=42, n_jobs=-1, refit=True, verbose=1
)
rnd_search.fit(X_train, y_train)

print(f"\nBest params:  {rnd_search.best_params_}")
print(f"Best ROC-AUC: {rnd_search.best_score_:.4f}")

In [ ]:
# Search results visualisation
results_df = pd.DataFrame(rnd_search.cv_results_)
results_df = results_df.sort_values("mean_test_score", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(
    range(len(results_df)),
    results_df["mean_test_score"],
    yerr=results_df["std_test_score"],
    fmt="o", markersize=4, alpha=0.7, color="#3498db"
)
ax.axhline(results_df["mean_test_score"].iloc[0], color="red", linestyle="--",
           label=f"Best: {results_df['mean_test_score'].iloc[0]:.4f}")
ax.set_xlabel("Parameter combination (sorted by score)")
ax.set_ylabel("ROC-AUC (5-fold CV)")
ax.set_title("RandomizedSearchCV — 30 iterations", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "04_hyperparameter_search.png", bbox_inches="tight")
plt.show()

## 5. Final Evaluation on Test Set

> **Book principle:** Evaluate on the test set **once** — at the very end. Never tune after peeking at test results.

In [ ]:
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, RocCurveDisplay, classification_report
)

best_model = rnd_search.best_estimator_
y_pred      = best_model.predict(X_test)
y_proba     = best_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "roc_auc":   round(roc_auc_score(y_test, y_proba), 4),
    "accuracy":  round(accuracy_score(y_test, y_pred), 4),
    "f1":        round(f1_score(y_test, y_pred), 4),
    "precision": round(precision_score(y_test, y_pred), 4),
    "recall":    round(recall_score(y_test, y_pred), 4),
}

print("Final Test Set Metrics:")
print("=" * 35)
for metric, value in test_metrics.items():
    print(f"  {metric:<12}: {value:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

In [ ]:
# Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["No Disease", "Disease"],
            yticklabels=["No Disease", "Disease"])
axes[0].set_title("Confusion Matrix — Test Set", fontweight="bold")
axes[0].set_ylabel("True Label")
axes[0].set_xlabel("Predicted Label")

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], color="#e74c3c")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random classifier")
axes[1].set_title("ROC Curve — Test Set", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / "04_confusion_roc.png", bbox_inches="tight")
plt.show()

## 6. Feature Importance — What Drives the Predictions?

In [ ]:
# Use the full pipeline from Notebook 03 if available, else use raw feature names
pipeline_path = INTERIM / "preprocessing_pipeline.joblib"
if pipeline_path.exists():
    preproc = joblib.load(pipeline_path)
    try:
        feature_names = preproc.named_steps["preprocessor"].get_feature_names_out().tolist()
    except Exception:
        feature_names = [f"f_{i}" for i in range(X_train.shape[1])]
else:
    feature_names = [f"feature_{i}" for i in range(X_train.shape[1])]

if len(feature_names) == len(best_model.feature_importances_):
    imp_df = pd.DataFrame({
        "feature": feature_names,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=imp_df, x="importance", y="feature", palette="RdYlGn_r", ax=ax)
    ax.set_title("Top 15 Feature Importances — Tuned Random Forest",
                 fontweight="bold", fontsize=12)
    ax.set_xlabel("Mean Decrease Impurity")
    plt.tight_layout()
    plt.savefig(FIGURES / "04_feature_importance_final.png", bbox_inches="tight")
    plt.show()

    print("\nTop 10 features driving cardiovascular risk prediction:")
    print(imp_df.head(10).to_string(index=False))

## 7. Threshold Analysis — Precision / Recall Trade-off

In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores_thresh = 2 * precisions * recalls / (precisions + recalls + 1e-9)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions[:-1], "b--", label="Precision")
ax.plot(thresholds, recalls[:-1],    "g-",  label="Recall")
ax.plot(thresholds, f1_scores_thresh[:-1], "r-", label="F1-Score", alpha=0.7)

# Mark default threshold (0.5)
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.6, label="Default threshold (0.5)")

# Mark max F1 threshold
best_thresh_idx = np.argmax(f1_scores_thresh[:-1])
best_thresh = thresholds[best_thresh_idx]
ax.axvline(best_thresh, color="red", linestyle="--", alpha=0.6,
           label=f"Max F1 threshold ({best_thresh:.2f})")

ax.set_xlabel("Decision Threshold", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Precision / Recall / F1 vs Decision Threshold", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(FIGURES / "04_threshold_analysis.png", bbox_inches="tight")
plt.show()

print(f"\nMax-F1 threshold: {best_thresh:.3f}")
print(f"At this threshold — Precision: {precisions[best_thresh_idx]:.3f}  Recall: {recalls[best_thresh_idx]:.3f}")

## 8. Business Translation — Revenue Impact

In [ ]:
model_accuracy = test_metrics["accuracy"]

# Revenue tier table from Notebook 00
tiers = [
    (0.50, 0.55, 0.05),
    (0.55, 0.60, 0.10),
    (0.60, 0.65, 0.15),
    (0.65, 0.70, 0.20),
    (0.70, 0.75, 0.25),
    (0.75, 1.00, 0.25),
]

fee_pct = 0.0
for lo, hi, pct in tiers:
    if lo <= model_accuracy < hi:
        fee_pct = pct
        break

n_patients_month  = 1_000
cost_per_patient  = 1_000  # USD
baseline_accuracy = 0.65
baseline_fee_pct  = 0.15

model_revenue_per_pt   = cost_per_patient * fee_pct
baseline_revenue_per_pt = cost_per_patient * baseline_fee_pct

model_revenue_monthly   = model_revenue_per_pt * n_patients_month
baseline_revenue_monthly = baseline_revenue_per_pt * n_patients_month
revenue_uplift = model_revenue_monthly - baseline_revenue_monthly

print("=" * 50)
print(f"MODEL ACCURACY            : {model_accuracy:.1%}")
print(f"FEE TIER ACHIEVED         : {fee_pct:.0%} of \$1,000")
print(f"REVENUE PER PATIENT       : \${model_revenue_per_pt:,.0f}")
print("-" * 50)
print(f"BASELINE REVENUE/MONTH    : \${baseline_revenue_monthly:,.0f} ({n_patients_month:,} pts × \${baseline_revenue_per_pt})")
print(f"MODEL REVENUE/MONTH       : \${model_revenue_monthly:,.0f} ({n_patients_month:,} pts × \${model_revenue_per_pt})")
print(f"MONTHLY REVENUE UPLIFT    : \${revenue_uplift:,.0f} ({revenue_uplift/baseline_revenue_monthly:+.0%})")
print(f"ANNUAL REVENUE UPLIFT     : \${revenue_uplift*12:,.0f}")
print("=" * 50)

In [ ]:
# Business impact chart
scenarios = {
    f"Baseline\n({baseline_accuracy:.0%})": baseline_revenue_monthly / 1e3,
    f"Our Model\n({model_accuracy:.1%})": model_revenue_monthly / 1e3,
}

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(scenarios.keys(), scenarios.values(),
              color=["#95a5a6", "#27ae60"], edgecolor="white", linewidth=1.5)
ax.set_ylabel("Monthly Revenue (thousands USD)", fontsize=11)
ax.set_title("Baseline vs ML Model — Monthly Revenue\n(1,000 patients)",
             fontsize=12, fontweight="bold")
for bar, val in zip(bars, scenarios.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"\${val:.0f}K", ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.annotate(
    f"+\${revenue_uplift/1e3:.0f}K/month",
    xy=(1, model_revenue_monthly/1e3), xytext=(1.3, (model_revenue_monthly + baseline_revenue_monthly)/2e3),
    fontsize=11, color="#27ae60", fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="#27ae60")
)
plt.tight_layout()
plt.savefig(FIGURES / "04_business_impact.png", bbox_inches="tight")
plt.show()

## 9. Save Final Model & Metrics

In [ ]:
MODELS_DIR   = PROJECT_ROOT / "models"
REPORTS_DIR  = PROJECT_ROOT / "reports"
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

# Save model
joblib.dump(best_model, MODELS_DIR / "model.joblib")
print(f"Model saved to {MODELS_DIR / 'model.joblib'}")

# Save metrics
full_metrics = {
    **test_metrics,
    "model_selected": best_name if 'best_name' in dir() else "random_forest_tuned",
    "best_params": {k: str(v) for k, v in rnd_search.best_params_.items()},
    "cv_best_score": round(rnd_search.best_score_, 4),
    "business": {
        "model_accuracy": model_accuracy,
        "fee_tier_pct": fee_pct,
        "revenue_per_patient_usd": model_revenue_per_pt,
        "monthly_revenue_uplift_usd": revenue_uplift,
        "annual_revenue_uplift_usd": revenue_uplift * 12,
    }
}

metrics_path = REPORTS_DIR / "metrics.json"
metrics_path.write_text(json.dumps(full_metrics, indent=2))
print(f"Metrics saved to {metrics_path}")
print()
print(json.dumps(full_metrics, indent=2))

## 10. Summary

| Step | Result |
|---|---|
| **Baseline (Logistic Regression)** | ROC-AUC ≈ 0.79 |
| **Best model (cross-validation)** | Random Forest — highest mean ROC-AUC |
| **After RandomizedSearchCV** | ROC-AUC improved via tuning |
| **Test set evaluation** | Single, clean evaluation — no overfitting to eval set |
| **Feature importance** | ap_hi, age_years, bmi are the top predictors — consistent with EDA |
| **Business impact** | Model achieves higher accuracy tier → measurable revenue uplift |

**Next → Notebook 05: Deployment & Consumption**